In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
        .appName("Gold")

        # Delta Lake
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

        # Memory (CRITICAL for 100M+ rows)
        .config("spark.driver.memory", "8g")
        .config("spark.executor.memory", "8g")

        # CPU cores
        .config("spark.driver.cores", "4")

        # Shuffle tuning
        .config("spark.sql.shuffle.partitions", "200")

        # Adaptive query optimization
        .config("spark.sql.adaptive.enabled", "true")

        # Prevent huge driver result crashes
        .config("spark.driver.maxResultSize", "2g")

        # UI tuning
        .config("spark.ui.showConsoleProgress", "false")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [2]:
#importing modules
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os
import logging
import requests

In [3]:
log_dir = "../logs"
os.makedirs(log_dir, exist_ok=True)

logging.basicConfig(
    filename=os.path.join(log_dir, "03_silver_to_gold.log"),
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

In [4]:
logging.info(f"Processing Silver Layer")

In [5]:
df_trip_data = spark.read\
                        .format("delta") \
                        .load(r"..\02_storage_silver\yellow_taxi")

In [6]:
df_trip_data.count()

113266731

### Dimensional Modelling Part 1 - Create Dimensional Tables

From Data Dictionary

In [7]:
vendor_data = [
    (1, "Creative Mobile Technologies"),
    (2, "Curb Mobility"),
    (6, "Myle Technologies"),
    (7, "Helix")
]

dim_vendor = spark.createDataFrame(vendor_data, ["vendor_id", "vendor_name"])

In [8]:
rate_data = [
    (1, "Standard"),
    (2, "JFK"),
    (3, "Newark"),
    (4, "Nassau/Westchester"),
    (5, "Negotiated fare"),
    (6, "Group ride"),
    (99, "Unknown")
]

dim_rate = spark.createDataFrame(rate_data, ["rate_code_id", "rate_name"])

In [9]:
payment_data = [
    (0, "Flex Fare"),
    (1, "Credit Card"),
    (2, "Cash"),
    (3, "No charge"),
    (4, "Dispute"),
    (5, "Unknown"),
    (6, "Voided trip")
]

dim_payment = spark.createDataFrame(payment_data, ["payment_type", "payment_name"])

In [10]:
store_flag_data = [
    ("Y", "Stored and forwarded"),
    ("N", "Sent directly")
]

dim_store_flag = spark.createDataFrame(
    store_flag_data,
    ["store_and_fwd_flag", "store_flag_desc"]
)

Trip Zone Lookup

In [11]:
url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
local_path = r"..\00_storage_raw\taxi_zone_lookup\taxi_zone_lookup.csv"
os.makedirs(os.path.dirname(local_path), exist_ok=True)

if os.path.exists(local_path):
    logging.info("taxi_zone_lookup.csv already exists. Skipping download.")
else:
    logging.info("taxi_zone_lookup.csv not found. Downloading...")

    response = requests.get(url)

    if response.status_code == 200:
        with open(local_path, "wb") as f:
            f.write(response.content)

        logging.info("Location dimension pulled from source API")
    else:
        logging.error(f"Failed to download file. Status code: {response.status_code}")

In [12]:
dim_location = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(local_path)
)
dim_location.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [13]:
dim_location = dim_location.select(
    col("LocationID").alias("location_id"),
    col("Borough").alias("borough"),
    col("Zone").alias("zone"),
    col("service_zone")
)

Write to dim folder

In [14]:
def write_dim(df, name):
    (
        df
        .coalesce(1)
        .write
        .format("delta")
        .mode("overwrite")
        .save(fr"..\03_storage_gold\dimensions\{name}")
    )
    logging.info(f"{name} dimension written to Gold layer")

write_dim(dim_vendor, "dim_vendor")
write_dim(dim_payment, "dim_payment_type")
write_dim(dim_rate, "dim_rate_code")
write_dim(dim_store_flag, "dim_store_forward_flag")
write_dim(dim_location, "dim_location")

### Dimensional Modelling Part 2- Build Fact Tables

Tables for dashboard

In [15]:
fact_trip_yearly_summary = (
    df_trip_data
    .groupBy("trip_year")
    .agg(
        count("*").alias("total_trips"),

        sum("total_amount").alias("total_revenue"),
        avg("total_amount").alias("avg_trip_value"),

        sum("trip_distance").alias("total_distance"),
        avg("trip_distance").alias("avg_distance"),

        avg("fare_per_mile").alias("avg_fare_per_mile"),

        avg("passenger_count").alias("avg_passengers"),

        sum("airport_fee").alias("total_airport_fee"),
        sum("congestion_surcharge").alias("total_congestion_fee"),
        sum("cbd_congestion_fee").alias("total_cbd_fee")
    )
)

fact_trip_yearly_summary.show()

+---------+-----------+--------------------+------------------+--------------------+------------------+------------------+------------------+-----------------+--------------------+-------------+
|trip_year|total_trips|       total_revenue|    avg_trip_value|      total_distance|      avg_distance| avg_fare_per_mile|    avg_passengers|total_airport_fee|total_congestion_fee|total_cbd_fee|
+---------+-----------+--------------------+------------------+--------------------+------------------+------------------+------------------+-----------------+--------------------+-------------+
|     2025|   41129662|1.2081315178503685E9|29.373728328970184|  1.48191636029903E8|3.6030355909538714| 7.709257674179114| 1.240005157348485|        5104525.5|       7.516083875E7|2.272175075E7|
|     2023|   35010575|1.0309443996965344E9|29.446657179910197| 1.273132104599083E8|3.6364215800485513|7.4820951261134905|1.3837177767003255|       5034845.95|        7.91057475E7|          0.0|
|     2024|   37126494|1.

In [16]:
fact_payment_summary = (
    df_trip_data
    .groupBy("trip_year", "payment_type")
    .agg(
        count("*").alias("total_trips"),

        sum("total_amount").alias("total_revenue"),
        avg("total_amount").alias("avg_trip_value"),

        avg("fare_per_mile").alias("avg_fare_per_mile")
    )
)

fact_payment_summary.show()

+---------+------------+-----------+--------------------+------------------+------------------+
|trip_year|payment_type|total_trips|       total_revenue|    avg_trip_value| avg_fare_per_mile|
+---------+------------+-----------+--------------------+------------------+------------------+
|     2025|           2|    3793029| 9.765742731000519E7|25.746554352736347| 7.978644020385868|
|     2025|           3|      64107|  1596042.8200000029| 24.89654515107559| 8.179160154117326|
|     2025|           1|   28923661| 8.839763788505435E8|30.562395916981032|7.7573899873845065|
|     2025|           0|    8304301|2.2362990123000568E8| 26.92940696995517| 7.413858387358387|
|     2025|           4|      44563|  1271695.6500000025|28.537029598545935|7.9114467158853845|
|     2025|           5|          1|               71.99|             71.99|              5.18|
|     2023|           4|      31804|   860577.0000000013|27.058766192931746| 7.558567790215074|
|     2023|           1|   28248644| 8.5

In [17]:
fact_location_summary = (
    df_trip_data
    .groupBy("trip_year", "pu_location_id")
    .agg(
        count("*").alias("total_trips"),

        sum("total_amount").alias("total_revenue"),

        avg("total_amount").alias("avg_trip_value"),
        avg("trip_distance").alias("avg_distance"),

        avg("fare_per_mile").alias("avg_fare_per_mile")
    )
)

fact_location_summary.show()

+---------+--------------+-----------+--------------------+------------------+------------------+------------------+
|trip_year|pu_location_id|total_trips|       total_revenue|    avg_trip_value|      avg_distance| avg_fare_per_mile|
+---------+--------------+-----------+--------------------+------------------+------------------+------------------+
|     2025|           163|    1067505|2.8246651160000786E7|26.460439211058297|2.5279080566367473| 8.743824516044596|
|     2025|           237|    1825997| 3.998169039999788E7| 21.89581384854295|  1.90153014490168| 8.765104712658292|
|     2025|            10|      19576|  1105724.4700000023| 56.48367746219873|10.001103391908462|5.3614997956681645|
|     2025|            52|      13504|   428277.6400000001|31.714872630331758| 5.260289543838866| 6.268162766587677|
|     2025|            59|        238|             6845.01|28.760546218487395|6.0399159663865545|5.4075210084033625|
|     2025|           214|        290|            12129.26|41.82

In [18]:
df_with_distance_bucket = (
    df_trip_data
    .withColumn(
        "distance_bucket",

        when(col("trip_distance") < 2, "short")
        .when(col("trip_distance") < 10, "medium")
        .otherwise("long")
    )
)

fact_distance_summary = (
    df_with_distance_bucket
    .groupBy("trip_year", "distance_bucket")
    .agg(
        count("*").alias("total_trips"),

        sum("total_amount").alias("total_revenue"),

        avg("total_amount").alias("avg_trip_value"),

        avg("fare_per_mile").alias("avg_fare_per_mile"),

        avg("trip_distance").alias("avg_distance")
    )
)

fact_distance_summary.show()

+---------+---------------+-----------+--------------------+------------------+-----------------+------------------+
|trip_year|distance_bucket|total_trips|       total_revenue|    avg_trip_value|avg_fare_per_mile|      avg_distance|
+---------+---------------+-----------+--------------------+------------------+-----------------+------------------+
|     2025|         medium|   17029617| 5.532908846499877E8| 32.48991945326708|6.185755854637255| 4.150381212916651|
|     2025|          short|   20713888|3.7881350715958625E8| 18.28789974917245| 9.55872988643376|1.2046562074681766|
|     2025|           long|    3386157| 2.760271260401979E8| 81.51634021700644|4.057593626048607|15.521761043565911|
|     2023|          short|   18320287|3.1824397894099486E8| 17.37112409543556|9.071385029603757|1.2062346588785153|
|     2023|         medium|   13472690| 4.368442126301617E8|32.424423973991956|6.124587094335013| 4.002731351346512|
|     2023|           long|    3217598|2.7585620813024473E8| 85.

In [19]:
fact_trip_time_summary = (
    df_trip_data
    .groupBy(
        "trip_year",
        "pickup_month",
        "pickup_day",
        "pickup_hour"
    )
    .agg(
        count("*").alias("total_trips"),

        sum("total_amount").alias("total_revenue"),
        avg("total_amount").alias("avg_trip_value"),

        avg("trip_distance").alias("avg_distance"),

        avg("fare_per_mile").alias("avg_fare_per_mile"),

        avg("passenger_count").alias("avg_passengers")
    )
)

fact_trip_time_summary.show()

+---------+------------+----------+-----------+-----------+------------------+------------------+------------------+------------------+------------------+
|trip_year|pickup_month|pickup_day|pickup_hour|total_trips|     total_revenue|    avg_trip_value|      avg_distance| avg_fare_per_mile|    avg_passengers|
+---------+------------+----------+-----------+-----------+------------------+------------------+------------------+------------------+------------------+
|     2025|           5|         1|         17|      29723| 914147.4400000072| 30.75555764895896|3.9163169935739925| 7.675907882784365|1.2937119402482926|
|     2025|           5|         1|         20|      22346| 680453.6400000028|30.450802828246793|  4.25103239953459| 6.546837465318165|1.2923118231450819|
|     2025|           2|         3|         23|      11595|339530.75000000093|29.282514014661572|4.1076670978870276| 6.141713669685203|1.2011211729193618|
|     2025|           1|         7|          3|       6399| 151146.399

In [20]:
fact_vendor_summary = (
    df_trip_data
    .groupBy(
        "trip_year",
        "vendor_id"
    )
    .agg(
        count("*").alias("total_trips"),

        sum("total_amount").alias("total_revenue"),

        avg("total_amount").alias("avg_trip_value"),

        avg("trip_distance").alias("avg_distance"),

        avg("fare_per_mile").alias("avg_fare_per_mile"),

        avg("passenger_count").alias("avg_passengers"),

        sum("airport_fee").alias("total_airport_fee"),

        sum("congestion_surcharge").alias("total_congestion_fee"),

        sum("cbd_congestion_fee").alias("total_cbd_fee")
    )
)

fact_vendor_summary.show()

+---------+---------+-----------+--------------------+------------------+------------------+------------------+------------------+-----------------+--------------------+-------------+
|trip_year|vendor_id|total_trips|       total_revenue|    avg_trip_value|      avg_distance| avg_fare_per_mile|    avg_passengers|total_airport_fee|total_congestion_fee|total_cbd_fee|
+---------+---------+-----------+--------------------+------------------+------------------+------------------+------------------+-----------------+--------------------+-------------+
|     2025|        2|   32248863| 9.584266188704474E8|29.719702641003106| 3.597903500348383| 7.742492326937115|1.2686642936837804|        4328251.5|       5.961897325E7|1.818548075E7|
|     2025|        6|       1688|   58593.01000000001| 34.71149881516588|  9.73664691943128|1.0870023696682465|               1.0|              0.0|                 0.0|          0.0|
|     2025|        1|    8879111|2.4964630597003302E8|28.116137524357228| 3.6205

Write to Disk

In [21]:
def write_fact(df, name):
    (
        df
        .coalesce(1)
        .write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .save(fr"..\03_storage_gold\facts\{name}")
    )
    logging.info(f"{name}  written to Gold layer")

In [22]:
write_fact(fact_trip_yearly_summary, "fact_trip_yearly_summary")
write_fact(fact_payment_summary, "fact_payment_summary")
write_fact(fact_location_summary, "fact_location_summary")
write_fact(fact_distance_summary, "fact_distance_summary")
write_fact(fact_trip_time_summary, "fact_trip_time_summary")
write_fact(fact_vendor_summary, "fact_vendor_summary")